# Task 5 — Spark Structured Streaming → MongoDB (Member 3)

**Ngày chạy:** 25/07/2026  
**Môi trường:** Docker Compose; Spark 3.5.1; Kafka 7.8.0; MongoDB 7.0.12; MongoDB Spark Connector 10.7.0.

Notebook này ghi lại một lần chạy end-to-end thật: Kafka `cpg.metadata` → Spark Structured Streaming → MongoDB `cpg.metadata`. Output bên dưới được lấy từ container và checkpoint của lần chạy, không phải output minh họa.

## 1. Approach và cấu trúc Spark job

Pipeline nằm trong `src/metadata_streaming_job.py`:

1. `SparkSession` được cấu hình MongoDB connector.
2. `readStream.format("kafka")` subscribe topic `cpg.metadata`, đọc từ `earliest`, bật `failOnDataLoss` và Kafka headers.
3. Kafka `value` (binary) được cast sang string rồi parse bằng `from_json` với `StructType` tường minh. Tất cả trường bắt buộc và nested edge counts được kiểm tra khác null; record sai topic/schema bị loại.
4. Cột `_id` được gán bằng `file_id`.
5. `writeStream.format("mongodb")` ghi collection `cpg.metadata` với `operationType=replace`, `idFieldList=_id`, `upsertDocument=true`, `writeConcern.w=majority` và checkpoint bền vững.

**Upsert key:** `_id = file_id`. `file_id` do parser tạo ổn định theo repository và file, nên replay/cập nhật cùng file sẽ replace document hiện tại thay vì sinh bản sao. Không dùng riêng `file_path`, vì hai repository có thể có cùng đường dẫn.

In [1]:
# Schema thực tế do build_metadata_schema() tạo
from src.metadata_streaming_job import build_metadata_schema
print(build_metadata_schema().treeString())

root
 |-- schema_version: string (nullable = false)
 |-- event_time: string (nullable = false)
 |-- topic: string (nullable = false)
 |-- repo_id: string (nullable = false)
 |-- file_id: string (nullable = false)
 |-- file_path: string (nullable = false)
 |-- file_hash: string (nullable = false)
 |-- parse_status: string (nullable = false)
 |-- file_size_bytes: long (nullable = false)
 |-- total_nodes: long (nullable = false)
 |-- total_edges: struct (nullable = false)
 |    |-- ast: long (nullable = false)
 |    |-- cfg: long (nullable = false)
 |    |-- dfg: long (nullable = false)
 |    |-- call: long (nullable = false)
 |-- parser_version: string (nullable = false)
 |-- parse_duration_ms: double (nullable = false)


In [2]:
from pprint import pprint
from src.spark_mongo_sink import build_mongo_write_config
pprint(build_mongo_write_config('mongodb://mongodb:27017', 'cpg', 'metadata'))

{'spark.mongodb.write.connection.uri': 'mongodb://mongodb:27017',
 'spark.mongodb.write.database': 'cpg',
 'spark.mongodb.write.collection': 'metadata',
 'spark.mongodb.write.operationType': 'replace',
 'spark.mongodb.write.idFieldList': '_id',
 'spark.mongodb.write.upsertDocument': 'true',
 'spark.mongodb.write.writeConcern.w': 'majority'}


## 2. Lệnh chạy thật

```powershell
docker compose --profile person3 run -d --name cpg-task5-spark-run spark-person3 `
  --master local[1] `
  --conf spark.jars.ivy=/opt/ivy `
  --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1,org.mongodb.spark:mongo-spark-connector_2.12:10.7.0 `
  /opt/project/src/metadata_streaming_job.py `
  --brokers kafka:29092 --topic cpg.metadata `
  --mongo-uri mongodb://mongodb:27017 --mongo-db cpg --mongo-collection metadata `
  --checkpoint-location /opt/checkpoints/task5-member3-20260725
```

Kafka có 5 record ở offsets `{partition 0: 2, partition 1: 3, partition 2: 0}`. Ba `file_id` ổn định xuất hiện trong các record; vì replace/upsert nên MongoDB còn 3 current-state documents.

In [3]:
# Trích log thật: docker logs cpg-task5-spark-run
print('Xem output đã lưu của micro-batch 0 bên dưới.')

26/07/25 11:52:45 INFO WriteToDataSourceV2Exec: Start processing MicroBatchWrite[epoch: 0, writer: MongoStreamingWrite].
26/07/25 11:52:49 INFO KafkaBatchReaderFactory: topicPartition=cpg.metadata-1 fromOffset=0 untilOffset=3 batchId=0
26/07/25 11:53:25 INFO KafkaDataConsumer: topicPartition=cpg.metadata-0 read 2 records
26/07/25 11:53:25 INFO WriteToDataSourceV2Exec: MongoStreamingWrite epoch 0 is committing.
26/07/25 11:53:25 INFO WriteToDataSourceV2Exec: MongoStreamingWrite epoch 0 committed.
26/07/25 11:53:26 INFO CheckpointFileManager: commits/0 written atomically
26/07/25 11:53:26 INFO MicroBatchExecution: Streaming query made progress
queryId: 20700212-9644-4d19-a4ed-a514b692946c
batchId: 0
numInputRows: 5
endOffset: {cpg.metadata: {0: 2, 1: 3, 2: 0}}
maxOffsetsBehindLatest: 0
triggerExecution: 44925 ms


In [4]:
# Output thật từ mongosh trong container MongoDB
# db.getSiblingDB('cpg').metadata.countDocuments({})
# db.getSiblingDB('cpg').metadata.aggregate([{$group:{_id:'$file_id',n:{$sum:1}}},{$match:{n:{$gt:1}}}])
# db.getSiblingDB('cpg').metadata.findOne({_id:'file_task5_app'})
print('Kết quả truy vấn được lưu trong output của cell.')

document_count: 3
duplicate_file_id_groups: []
{
  "_id": "file_task5_app",
  "schema_version": "1.0",
  "event_time": "2026-07-25T12:00:01Z",
  "topic": "cpg.metadata",
  "repo_id": "demo/task5",
  "file_id": "file_task5_app",
  "file_path": "src/app.py",
  "file_hash": "sha256_app_v1",
  "parse_status": "success",
  "file_size_bytes": 420,
  "total_nodes": 23,
  "total_edges": {"ast": 18, "cfg": 7, "dfg": 5, "call": 2},
  "parser_version": "ast-stdlib-1.0",
  "parse_duration_ms": 12.4
}


## 3. MongoDB collection

Collection thật tại thời điểm chạy: `mongodb://localhost:27017` → database `cpg` → collection `metadata`; có 3 documents: `file_task5_app`, `file_task5_model`, `file_task5_utils`.

### MongoDB Compass — collection `cpg.metadata`

Ảnh dưới đây xác nhận kết nối `localhost:27017`, database `cpg`, collection `metadata`, tổng cộng 3 documents (`1–3 of 3`). Document `file_task5_app` và `file_task5_utils` thể hiện các trường do Spark ghi, gồm `file_id`, `file_path`, `file_hash`, `parse_status` và các thống kê phân tích.

![MongoDB Compass — collection và document mẫu](../docs/images/task5_mongodb_compass.png)

Ảnh bổ sung hiển thị đầy đủ hơn hai document `file_task5_utils` và `file_task5_model`:

![MongoDB Compass — các document còn lại](../docs/images/task5_mongodb_compass_more_documents.png)

## 4. Reflection

Phần quan trọng nhất không chỉ là đọc Kafka rồi ghi MongoDB, mà là bảo đảm kết quả đúng khi job/repository event bị chạy lại. Spark checkpoint lưu offsets và batch commit, còn `_id = file_id` cùng replace/upsert làm database idempotent. Chỉ có checkpoint thì một replay ngoài checkpoint vẫn có thể tạo duplicate; chỉ có upsert thì Spark vẫn phải đọc lại dữ liệu tốn thời gian sau restart. Hai cơ chế bổ sung cho nhau.

Schema tường minh giúp phát hiện contract drift sớm và giữ kiểu của nested `total_edges`, thay vì để Spark suy luận khác nhau giữa micro-batches. Trade-off là record thiếu bất kỳ trường bắt buộc nào bị loại; production nên bổ sung dead-letter topic và metric đếm malformed records để không mất lỗi âm thầm.

Lần chạy này cũng cho thấy batch đầu chậm hơn (khoảng 44.9 giây) do cold start và nạp connector; các batch sau sẽ nhẹ hơn. Với tải lớn nên giới hạn `maxOffsetsPerTrigger`, theo dõi `processedRowsPerSecond`/offset lag, và đặt checkpoint trên storage bền vững thay vì filesystem cục bộ.

## 5. Kết luận kiểm chứng

- Spark job và connector đã khởi động thật.
- Micro-batch 0 đọc 5 Kafka records, lag cuối batch bằng 0.
- MongoDB commit thành công và checkpoint `commits/0` tồn tại.
- Collection có 3 documents hiện hành và không có nhóm `file_id` trùng.
- Hai ảnh MongoDB Compass xác nhận collection `cpg.metadata` chứa đủ ba document mẫu đã được đính kèm.